# 🚀 TikTok Brasil Turbo — Automação de Crescimento com IA
### Nichos Virais: **Direita (Bolsonaro)**, **Esquerda (Lula)**, **Tarot/Cartomante** e **Fé Católica**

Este notebook executa o pipeline de vídeos curtos em formato 9:16 com:
- 🧠 Roteiros via **Google Gemini Gratuito** (pool de até 15 API Keys com rotação automática).
- 🎙️ Vozes neurais em PT-BR (**XTTS v2 na GPU com amostras ElevenLabs** ou **EdgeTTS gratuito**).
- 📦 **Modo Individual ou BATCH via Google Drive**: processe uma pasta inteira de templates prontos de uma vez só!
- ☁️ **Exportação Limpa no Google Drive**: apenas o arquivo `.mp4` final e o `.txt` pronto para colar com **5 hashtags virais**.

## 1️⃣ Conectar ao Google Drive
Garante que todos os vídeos finais e textos de postagem fiquem salvos diretamente na sua conta do Google Drive.

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/TikTok_Producao'
TEMPLATES_DIR = '/content/drive/MyDrive/TikTok_Templates'
os.makedirs(DRIVE_DIR, exist_ok=True)
os.makedirs(TEMPLATES_DIR, exist_ok=True)
print(f"✅ Google Drive conectado!")
print(f"📁 Pasta de Produção: {DRIVE_DIR}")
print(f"📁 Pasta de Templates Batch: {TEMPLATES_DIR}")

## 2️⃣ Clonar Repositório e Instalar Dependências
Clona o **MPT-Brasil-Turbo** e instala FFmpeg, dependências de áudio, vídeo e IA.

In [ ]:
!nvidia-smi

import os, sys
REPO_URL = "https://github.com/lipiw/MPT-Brasil-Turbo.git"
REPO_DIR = "/content/MPT-Brasil-Turbo"

if not os.path.exists(REPO_DIR):
    !git clone $REPO_URL $REPO_DIR
else:
    !cd $REPO_DIR && git pull

if REPO_DIR not in sys.path:
    sys.path.append(REPO_DIR)

!apt-get -y update && apt-get -y install ffmpeg
!pip install -q edge-tts google-generativeai moviepy openai-whisper TTS torch torchvision

print("✅ Repositório clonado e ambiente configurado com sucesso!")

## 3️⃣ Execução em BATCH (Pasta do Drive com Templates)
Se você colocou arquivos `.txt` ou `.json` na pasta `TikTok_Templates` do seu Drive, selecione-a abaixo para rodar todos de uma só vez!

In [ ]:
#@title 📦 Processar Pasta em BATCH (Lote) { run: "auto" }
PASTA_TEMPLATES_DRIVE = "/content/drive/MyDrive/TikTok_Templates" #@param {type:"string"}
NICHO_PADRAO = "tarot" #@param ["tarot", "catolico", "direita", "esquerda"]
GEMINI_API_KEYS = "" #@param {type:"string"}

import sys, os
REPO_DIR = "/content/MPT-Brasil-Turbo"
if REPO_DIR not in sys.path:
    sys.path.append(REPO_DIR)

from app.services.batch_processor import BatchTemplateProcessor

if os.path.exists(PASTA_TEMPLATES_DRIVE) and os.listdir(PASTA_TEMPLATES_DRIVE):
    bp = BatchTemplateProcessor(drive_output_dir=DRIVE_DIR, gemini_keys_str=GEMINI_API_KEYS)
    resultados_batch = bp.process_folder(PASTA_TEMPLATES_DRIVE, default_nicho=NICHO_PADRAO)
    print(f"🎉 Batch concluído! {len(resultados_batch)} vídeos exportados para {DRIVE_DIR}")
else:
    print(f"ℹ️ Nenhum arquivo encontrado em {PASTA_TEMPLATES_DRIVE}. Use a Célula 4 para geração individual ou coloque seus templates lá!")

## 4️⃣ Execução Individual (1 Vídeo por Vez)
Gere um vídeo imediatamente a partir de um tema rápido.

In [ ]:
#@title 🎬 Gerar Vídeo Individual { run: "auto" }
NICHO = "tarot" #@param ["tarot", "catolico", "direita", "esquerda"]
TEMA = "Mensagem Urgente dos Seus Guias para Hoje" #@param {type:"string"}
GEMINI_API_KEYS = "" #@param {type:"string"}

import sys, os
REPO_DIR = "/content/MPT-Brasil-Turbo"
if REPO_DIR not in sys.path:
    sys.path.append(REPO_DIR)

from app.services.mpt_brasil_runner import run_nicho_generation

resultado = run_nicho_generation(
    nicho=NICHO,
    tema_personalizado=TEMA,
    drive_output_dir=DRIVE_DIR,
    gemini_keys_str=GEMINI_API_KEYS
)

print("\n--- 📝 CONTEÚDO PRONTO PARA POSTAGEM NO TIKTOK ---")
print(resultado['copy'])

## 5️⃣ Pré-visualização do Vídeo Renderizado
Assista diretamente no navegador.

In [ ]:
from IPython.display import HTML
from base64 import b64encode

video_path = resultado['video_path']
if os.path.exists(video_path) and os.path.getsize(video_path) > 1000:
    mp4 = open(video_path, 'rb').read()
    data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
    display(HTML(f"""
    <video width=360 height=640 controls autoplay loop>
          <source src="{data_url}" type="video/mp4">
    </video>
    """))
else:
    print(f"Arquivo de vídeo salvo em: {video_path}")